# Planetas y estrellas anfitrionas

Trabaja con todos los planetas que cumplen el filtro publicado: detección por tránsito hasta 2024, solución predeterminada, radio y periodo positivos sin banderas de límite. Esta selección permite comparar propiedades; no representa todos los planetas del universo.

**Pregunta:** ¿Cómo se relacionan el periodo orbital y el radio dentro de esta selección? ¿Cuántas filas representan planetas y cuántas anfitrionas distintas?

Abre este cuaderno desde el paquete completo del módulo. Ejecuta las celdas en orden en Jupyter o en el editor de cuadernos de tu entorno. Solo las figuras requieren Matplotlib. Estos cuadernos leen los archivos incluidos y no descargan información.


In [ ]:
from pathlib import Path
import importlib.util
import json

if 'BASE_MODULO' not in globals():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    BASE_MODULO = next((p for c in candidatos for p in (c, c / 'materiales/analisis-datos') if (p / 'codigo/analysis_data.py').is_file()), None)
    if BASE_MODULO is None:
        raise FileNotFoundError('Abre el cuaderno dentro del paquete completo del módulo.')
spec = importlib.util.spec_from_file_location('analisis_modulo', BASE_MODULO / 'codigo/analysis_data.py')
analisis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analisis)
verificacion = analisis.validate_bundle(BASE_MODULO)
nombre = 'exoplanetas'
filas = analisis.read_csv(BASE_MODULO / 'datos' / (nombre + '.csv'), nombre)
fuente = json.loads((BASE_MODULO / 'fuentes.json').read_text(encoding='utf-8'))['datasets'][nombre]
print('Filas:', len(filas))
print('Consulta:', fuente['url_consulta'])
print('Fecha de descarga:', fuente['fecha_descarga'])
print('Unidades:', fuente['unidades'])


## Inspecciona antes de interpretar

Cada fila tiene un significado, cada columna una unidad y cada dato una procedencia. La validación anterior compara el CSV con su original y comprueba hashes. Revisa las primeras filas sin inferir que representan estadísticamente toda la selección.


In [ ]:
print(json.dumps(filas[:5], ensure_ascii=False, indent=2))
print('Límites de uso:')
for limite in fuente['limites']:
    print('-', limite)
resultado = analisis.summarize(nombre, filas)
print(json.dumps(resultado, ensure_ascii=False, indent=2))


## Reproduce una comprobación

Comprueba el resultado con una operación visible. Conserva la unidad y el denominador al describirlo.


In [ ]:
anfitrionas = {}
for fila in filas:
    anfitrionas.setdefault(fila['hostname'], []).append(fila)
repetidas = {nombre: len(grupo) for nombre, grupo in anfitrionas.items() if len(grupo) > 1}
print('Anfitrionas con más de un planeta en esta selección:', len(repetidas))
print('No conviertas las filas de planetas en observaciones estelares independientes.')


## Genera y examina los gráficos

La celda guarda PNG y SVG. Revisa títulos, ejes, unidades, selección y valores faltantes. Usa el CSV como alternativa tabular y describe con palabras el patrón que observas.


In [ ]:
if globals().get('MOSTRAR_FIGURAS', True):
    from IPython.display import Image, display
    imagenes = analisis.create_figures(BASE_MODULO, only=nombre)
    display(Image(filename=str(imagenes[0])))


## Interpreta y contrasta

Para un gráfico de estrellas, separa una tabla por anfitriona y revisa discrepancias de parámetros entre referencias antes de elegir un valor. No promedies valores incompatibles por comodidad. Las cifras de st_teff y st_rad permiten explorar temperatura y tamaño; no equivalen a un diagrama completo de Hertzsprung–Russell.

Escribe un hallazgo que pueda comprobarse, una limitación y una pregunta siguiente. Cita la URL, la fecha de descarga y el filtro. Para refrescar datos, ejecuta conscientemente el comando `refrescar` explicado en `codigo/README.md`; el análisis offline conserva la versión descargada.
